In [ ]:
# INSTALAÇÕES (rode uma vez)
!pip install -q missingno rapidfuzz kagglehub openpyxl

# Exercício 1 — Valores Ausentes

In [ ]:
# IMPORTAÇÕES
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import missingno as msno

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
# CARREGAMENTO DOS DADOS
caminho = '/content/drive/MyDrive/Colab Notebooks/melb_data.csv'
if not os.path.exists(caminho):
    caminho = 'melb_data.csv'
df = pd.read_csv(caminho)
df.head()

In [ ]:
# DIAGNÓSTICO INICIAL
print(df.shape)
df.info()
print(df.isnull().sum()[lambda s: s > 0])
print('Duplicadas:', df.duplicated().sum())

In [ ]:
# RESUMO DE VALORES AUSENTES (QUANTIDADE E %)
ausentes = df.isnull().sum()
ausentes = ausentes[ausentes > 0]
resumo = pd.DataFrame({
    'Quantidade': ausentes,
    'Porcentagem (%)': (ausentes / len(df) * 100).round(2)
}).sort_values(by='Quantidade', ascending=False)
resumo

In [ ]:
# VISUALIZAÇÃO DO PADRÃO DE AUSÊNCIA
msno.matrix(df)
plt.show()

# As ausências se concentram em poucas colunas (BuildingArea, YearBuilt, CouncilArea e Car).
# BuildingArea e YearBuilt costumam faltar nas mesmas linhas (aparecem juntas no gráfico).

In [ ]:
# ESTRATÉGIA 1 — REMOÇÃO
df_remocao = df.copy()

# BuildingArea (47%) e YearBuilt (40%) têm ausência demais: descarto as colunas.
# Car (0,5%) e CouncilArea (10%) têm pouca ausência: descarto só as linhas.
df_remocao = df_remocao.drop(columns=['BuildingArea', 'YearBuilt'])
df_remocao = df_remocao.dropna(subset=['Car', 'CouncilArea'])
print(df_remocao.shape)

In [ ]:
# ESTRATÉGIA 2 — IMPUTAÇÃO
df_imputacao = df.copy()

# Numéricas -> mediana (não é puxada por valores extremos). Categórica -> moda.
for col in ['BuildingArea', 'YearBuilt', 'Car']:
    df_imputacao[col] = df_imputacao[col].fillna(df_imputacao[col].median())
df_imputacao['CouncilArea'] = df_imputacao['CouncilArea'].fillna(df_imputacao['CouncilArea'].mode()[0])
print(df_imputacao.shape, '| ausentes restantes:', df_imputacao.isnull().sum().sum())

In [ ]:
# COMPARAÇÃO ENTRE AS ESTRATÉGIAS
print('Linhas -> original:', len(df), '| remoção:', len(df_remocao), '| imputação:', len(df_imputacao))

cols = ['Car', 'BuildingArea', 'YearBuilt']
comparacao = pd.concat({
    'original': df[cols].agg(['mean', 'median', 'std']).T,
    'remoção': df_remocao[[c for c in cols if c in df_remocao]].agg(['mean', 'median', 'std']).T,
    'imputação': df_imputacao[cols].agg(['mean', 'median', 'std']).T,
}, axis=1)
display(comparacao)

# Distribuição de BuildingArea antes e depois (cortada no percentil 99 para enxergar melhor)
lim = df['BuildingArea'].quantile(0.99)
plt.hist(df['BuildingArea'].dropna().clip(upper=lim), bins=50, alpha=0.6, density=True, label='original')
plt.hist(df_imputacao['BuildingArea'].clip(upper=lim), bins=50, alpha=0.6, density=True, label='imputado')
plt.legend(); plt.title('BuildingArea'); plt.show()

# Apoio para a reflexão: média x mediana
print(df[['Price', 'Landsize']].agg(['mean', 'median']))

In [ ]:
# RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO

# 1) Coluna com mais da metade ausente: imputar ou descartar?
# Depende de quão importante a coluna é para a análise, de por que os dados faltam e de existir outra
# coluna com informação parecida. Com mais da metade vazia, a imputação "inventa" a maior parte da coluna,
# então em geral descarto. Só imputo se a coluna for essencial (de preferência criando uma coluna indicando
# que o valor estava ausente).

# 2) Price e Landsize: média e mediana são parecidas?
# Não. Os dois têm poucos valores muito altos (imóveis caros, terrenos grandes), que puxam a média para cima
# (veja a tabela acima). Landsize ainda tem muitos zeros. A mediana é mais segura, porque não é afetada pelos extremos.

# 3) Quando a ausência é, por si só, uma informação útil?
# Quando o valor falta por um motivo ligado ao que se estuda. Exemplo: quem não informa a renda num pedido de
# crédito pode ter um perfil de risco diferente; aqui, o vendedor que não informa o ano de construção pode estar
# vendendo um imóvel mais antigo. Nesses casos vale criar uma coluna "estava ausente" antes de imputar.

# Exercício 2 — Duplicatas

In [ ]:
# IMPORTAÇÕES
import os
import pandas as pd
import numpy as np

# CARREGAMENTO DOS DADOS
arquivo = 'KPMG_VI_New_raw_data_update_final.xlsx'
for c in [arquivo, f'/content/drive/MyDrive/{arquivo}', f'/content/drive/MyDrive/Colab Notebooks/{arquivo}']:
    if os.path.exists(c):
        arquivo = c
        break
else:
    from google.colab import files
    files.upload()   # envie o arquivo .xlsx

In [ ]:
# CONFERÊNCIA DO CABEÇALHO E DOS TIPOS
trans = pd.read_excel(arquivo, sheet_name='Transactions')
print(trans.columns.tolist()[:4])   # o cabeçalho não está na primeira linha

# Procuro a linha que contém "transaction_id" e releio a planilha a partir dela
bruto = pd.read_excel(arquivo, sheet_name='Transactions', header=None, nrows=10)
linha = next(i for i in range(len(bruto)) if 'transaction_id' in bruto.iloc[i].astype(str).tolist())
trans = pd.read_excel(arquivo, sheet_name='Transactions', header=linha)

trans['transaction_date'] = pd.to_datetime(trans['transaction_date'], errors='coerce')
print(trans.shape)
print(trans.dtypes)
trans.head()

In [ ]:
# DUPLICATAS EXATAS
print('Duplicatas exatas:', trans.duplicated().sum())

In [ ]:
# DUPLICATAS POR SUBCONJUNTO DE COLUNAS
# Escolha: mesmo cliente + mesmo produto + mesma data + mesmo preço/custo + mesma modalidade (online ou não)
# provavelmente é a mesma transação registrada duas vezes. O transaction_id fica de fora porque muda a cada registro.
subconjunto = ['customer_id', 'product_id', 'transaction_date', 'list_price', 'standard_cost', 'online_order']
print('Duplicatas pelo subconjunto:', trans.duplicated(subset=subconjunto).sum())
trans[trans.duplicated(subset=subconjunto, keep=False)].sort_values(subconjunto).head(10)

In [ ]:
# UNICIDADE DOS IDENTIFICADORES
print('transaction_id único?', trans['transaction_id'].is_unique)
print('IDs repetidos:', trans['transaction_id'].duplicated().sum(), '| nulos:', trans['transaction_id'].isna().sum())

In [ ]:
# ESTRATÉGIA DE TRATAMENTO E IMPACTO
# Estratégia: manter o registro mais recente de cada grupo duplicado e remover os demais.
limpo = trans.sort_values('transaction_date').drop_duplicates(subset=subconjunto, keep='last').sort_index()

print('Registros removidos:', len(trans) - len(limpo))
impacto = pd.DataFrame({
    'antes': [len(trans), trans['list_price'].sum(), trans['list_price'].mean()],
    'depois': [len(limpo), limpo['list_price'].sum(), limpo['list_price'].mean()],
}, index=['linhas', 'soma list_price', 'média list_price'])
impacto['diferença'] = impacto['depois'] - impacto['antes']
impacto

In [ ]:
# RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO

# 1) Zero duplicatas exatas garante que não há registros repetidos?
# Não. A duplicata exata exige que todas as colunas sejam iguais, inclusive o transaction_id. Se o sistema
# gera um ID novo a cada registro, a mesma transação aparece duas vezes com IDs diferentes e o duplicated()
# não detecta. Por isso também verifico por subconjunto de colunas (acima).

# 2) Duas linhas idênticas são sempre um erro?
# Não. Um cliente pode comprar dois itens iguais no mesmo dia, pelo mesmo preço, em dois registros
# separados. Se a base não tem quantidade nem horário, as linhas ficam idênticas mesmo sendo vendas reais.
# Remover essas linhas faria o faturamento parecer menor do que é.

# 3) Por que escolher as colunas que definem "o mesmo registro" é uma decisão de negócio?
# Porque quem sabe o que conta como uma transação única é o negócio (um item? um carrinho?), e o erro
# custa diferente de cada lado: remover vendas reais diminui o faturamento, manter duplicadas aumenta.
# No CRISP-DM isso se define na etapa de Entendimento do Negócio, antes de aplicar o drop_duplicates.

# Exercício 3 — Padronização e Escalonamento

In [ ]:
# IMPORTAÇÕES
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# NOMES DAS COLUNAS (documentação do dataset, arquivo adult.names)
colunas = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'income']

In [ ]:
# CARREGAMENTO DOS DADOS
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
try:
    df = pd.read_csv(url, header=None, names=colunas)
except Exception:
    if not os.path.exists('adult.data'):
        from google.colab import files
        files.upload()   # envie o adult.data (baixe em archive.ics.uci.edu/dataset/2/adult)
    df = pd.read_csv('adult.data', header=None, names=colunas)
print(df.shape)
df.head()

In [ ]:
# INSPEÇÃO DAS COLUNAS CATEGÓRICAS
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Ausentes pelo isna():', df.isna().sum().sum())

for c in cat_cols:
    print(f'\n{c} ({df[c].nunique()} categorias):', df[c].unique()[:8].tolist())

print('\nTextos com espaço no início/fim:', {c: int((df[c] != df[c].str.strip()).sum()) for c in cat_cols})
print('Marcador "?" por coluna:', {c: int((df[c].str.strip() == '?').sum()) for c in cat_cols if (df[c].str.strip() == '?').any()})

In [ ]:
# PADRONIZAÇÃO DOS TEXTOS (ANTES x DEPOIS)
antes = df[cat_cols].nunique()
for c in cat_cols:
    df[c] = df[c].str.strip().str.lower()
depois = df[cat_cols].nunique()
pd.DataFrame({'categorias antes': antes, 'categorias depois': depois})

In [ ]:
# CONVERSÃO DOS MARCADORES DE AUSÊNCIA
df[cat_cols] = df[cat_cols].replace('?', np.nan)
novos = df.isna().sum()
print('Valores ausentes que "apareceram":')
print(novos[novos > 0])

In [ ]:
# ESCALONAMENTO — MIN-MAX
num_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
X = df[num_cols]
minmax = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=num_cols)
minmax.describe().T[['min', 'max', 'mean', 'std']]

In [ ]:
# ESCALONAMENTO — Z-SCORE
zscore = pd.DataFrame(StandardScaler().fit_transform(X), columns=num_cols)
zscore.describe().T[['min', 'max', 'mean', 'std']]

In [ ]:
# ESCALONAMENTO — ROBUSTO
robusto = pd.DataFrame(RobustScaler().fit_transform(X), columns=num_cols)
robusto.describe().T[['min', 'max', 'mean', 'std']]

In [ ]:
# COMPARAÇÃO ENTRE AS TÉCNICAS
comparacao = pd.concat({
    'original': X.describe().T[['min', 'max', 'mean', 'std']],
    'Min-Max': minmax.describe().T[['min', 'max', 'mean', 'std']],
    'Z-score': zscore.describe().T[['min', 'max', 'mean', 'std']],
    'Robusto': robusto.describe().T[['min', 'max', 'mean', 'std']],
}, axis=1)
display(comparacao)
print('% de zeros em capital-gain:', round((df['capital-gain'] == 0).mean() * 100, 1))

In [ ]:
# RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO

# 1) Por que isna().sum() retorna zero num dataset com dados faltando?
# Porque o pandas só reconhece como ausente o que é realmente nulo (NaN/None). Aqui a ausência vem escrita
# como o texto " ?" (com espaço), que para o pandas é um valor comum. Só depois de converter o "?" em NaN
# o isna() mostra os ausentes de workclass, occupation e native-country.

# 2) capital-gain (maioria zero): qual técnica é mais afetada?
# O Min-Max é o mais afetado: o maior valor (99.999) vira 1 e quase todos os outros ficam espremidos
# perto de 0. O Z-score também sofre (poucos valores gigantes elevam o desvio e geram z muito alto).
# O robusto usa mediana e IQR; como mais de 90% são zero, o IQR é 0 e a coluna quase não muda.

# 3) Faz sentido escalonar todas as colunas numéricas? O que é fnlwgt?
# Não. fnlwgt é o peso amostral do censo (quantas pessoas da população aquela linha representa), não uma
# característica da pessoa; não deve ser usada como variável do modelo nem escalonada como se fosse. Também
# não faz sentido escalonar education-num, que é só a versão numérica de education.

# Exercício 4 — Outliers

In [ ]:
# IMPORTAÇÕES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# CARREGAMENTO DOS DADOS
df = sns.load_dataset('diamonds')
num_cols = ['carat', 'depth', 'table', 'price', 'x', 'y', 'z']

In [ ]:
# ESTATÍSTICAS DESCRITIVAS E VALORES IMPOSSÍVEIS
display(df[num_cols].describe().T)

# x, y, z são dimensões em mm: não podem ser 0, e y > 20 ou z > 10 é impossível para esses diamantes
print('x, y ou z = 0:', (df[['x', 'y', 'z']] == 0).any(axis=1).sum())
print('y > 20:', (df['y'] > 20).sum(), '| z > 10:', (df['z'] > 10).sum())

In [ ]:
# DETECÇÃO — Z-SCORE
z = pd.DataFrame(np.abs(stats.zscore(df[num_cols])), columns=num_cols)
out_z = z > 3
print(out_z.sum())

In [ ]:
# DETECÇÃO — IQR
def limites_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

out_iqr = pd.DataFrame({c: (df[c] < limites_iqr(df[c])[0]) | (df[c] > limites_iqr(df[c])[1]) for c in num_cols})
pd.DataFrame({'Z-score': out_z.sum(), 'IQR': out_iqr.sum()})

In [ ]:
# VISUALIZAÇÕES
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(y=df[c], ax=ax)
    ax.set_title(c)
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.scatterplot(data=df, x='carat', y='price', alpha=0.2, ax=axes[0])
sns.scatterplot(data=df, x='x', y='y', alpha=0.3, ax=axes[1])
sns.scatterplot(data=df, x='y', y='z', alpha=0.3, ax=axes[2])
plt.tight_layout(); plt.show()

In [ ]:
# ERRO x VALOR EXTREMO LEGÍTIMO
# Critério: é ERRO o que é fisicamente impossível (dimensão 0, y > 20, z > 10).
# É valor extremo LEGÍTIMO o que está fora do IQR, mas é coerente (ex.: diamante grande, caro e com dimensões proporcionais).
erro = (df[['x', 'y', 'z']] == 0).any(axis=1) | (df['y'] > 20) | (df['z'] > 10)
legitimo = out_iqr.any(axis=1) & ~erro
print('Erros de registro:', erro.sum(), '| Extremos legítimos:', legitimo.sum())
df[erro]

In [ ]:
# TRATAMENTO — REMOÇÃO / CAPPING / TRANSFORMAÇÃO
sem_erro = df[~erro].copy()      # os erros saem em todas as estratégias

# Remoção: tira também as linhas fora do IQR em price
lo, hi = limites_iqr(sem_erro['price'])
df_remocao = sem_erro[(sem_erro['price'] >= lo) & (sem_erro['price'] <= hi)].copy()

# Capping: substitui os extremos pelos limites do IQR
df_capping = sem_erro.copy()
for c in num_cols:
    l, h = limites_iqr(sem_erro[c])
    df_capping[c] = df_capping[c].clip(l, h)

# Transformação: log e raiz quadrada do price
df_transf = sem_erro.copy()
df_transf['price_log'] = np.log1p(df_transf['price'])
df_transf['price_sqrt'] = np.sqrt(df_transf['price'])

In [ ]:
# COMPARAÇÃO ENTRE AS ESTRATÉGIAS
series = {'original': df['price'], 'remoção': df_remocao['price'], 'capping': df_capping['price'],
          'raiz quadrada': df_transf['price_sqrt'], 'log': df_transf['price_log']}
display(pd.DataFrame({n: s.agg(['count', 'mean', 'median', 'std', 'min', 'max', 'skew']) for n, s in series.items()}).T)

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, (n, s) in zip(axes.ravel(), series.items()):
    sns.histplot(s, bins=50, ax=ax); ax.set_title(f'price — {n}')
stats.probplot(df['price'], dist='norm', plot=axes.ravel()[-1]); axes.ravel()[-1].set_title('Q-Q plot de price')
plt.tight_layout(); plt.show()

print('Diamantes com mais de 4 quilates:', (df['carat'] > 4).sum())

In [ ]:
# RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO

# 1) Qual suposição o Z-score faz? Vale para price?
# Supõe que os dados seguem uma distribuição aproximadamente normal (em forma de sino), para que "mais de
# 3 desvios da média" signifique algo raro. Não vale para price: o histograma é muito assimétrico (cauda longa
# à direita) e o Q-Q plot se afasta da reta. Depois do log a forma fica bem mais próxima da normal.

# 2) Diamante com mais de 4 quilates: outlier a remover ou diamante raro?
# É um diamante raro, mas real (dimensões e preço coerentes). Se o objetivo é o preço do mercado comum, posso
# deixá-los de fora ou tratá-los à parte; se o objetivo é estudar joias de luxo, leilão ou seguro, eles são
# justamente o objeto de estudo e não podem ser removidos. Remove-se o erro de registro, não o raro.

# 3) Exemplo em que remover outliers esconderia o que o negócio quer descobrir.
# Detecção de fraude em cartão: a fraude é a transação fora do padrão. Remover outliers apagaria
# exatamente os casos que se quer encontrar. Vale também para falhas de máquinas e clientes "baleia".

# Exercício 5 — Projeto Integrado de Limpeza

In [ ]:
# IMPORTAÇÕES
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# PERGUNTA DE NEGÓCIO
# Em quais faixas de preço os produtos têm os maiores descontos médios, e como isso se relaciona com a avaliação dos clientes?

In [ ]:
# CARREGAMENTO DOS ARQUIVOS
pasta = '.'
try:
    import kagglehub
    pasta = kagglehub.dataset_download('oleksiimartusiuk/e-commerce-data-shein')
except Exception:
    print('Download do Kaggle falhou: envie os CSVs para o Colab (ícone de pasta à esquerda).')

arquivos = sorted(glob.glob(os.path.join(pasta, '**', '*.csv'), recursive=True)) or sorted(glob.glob('*.csv'))
for a in arquivos:
    print(f'{os.path.getsize(a)/1e6:7.1f} MB  {a}')

arquivos = arquivos[:2]      # dois arquivos (edite a lista para escolher outros)
brutos = {os.path.basename(a): pd.read_csv(a, low_memory=False) for a in arquivos}
for nome, d in brutos.items():
    print(nome, d.shape)

In [ ]:
# DIAGNÓSTICO INICIAL (REGISTRAR PROBLEMAS ANTES DE CORRIGIR)
MARCADORES = {'', 'n/a', 'na', 'nan', 'none', 'null', '-', '--', '?', 'unknown', 'undefined'}
ID_COL = re.compile(r'(^|_)(id|sku|url|link|code|zip|image|img)(_|$)', re.I)

def limpar_numero(s):
    return s.astype('string').str.replace(r'(R\$|US\$|[$€£%+\s])', '', regex=True)

def parece_numero(s):
    s = s.dropna()
    return 0.0 if s.empty else limpar_numero(s).str.fullmatch(r'-?\d[\d.,]*[kK]?').fillna(False).mean()

for nome, d in brutos.items():
    obj = d.select_dtypes(include='object').columns
    print(f'\n=== {nome} {d.shape} ===')
    print('Ausentes (%):'); print((d.isna().mean() * 100).round(1).sort_values(ascending=False).head(6))
    print('Duplicatas exatas:', d.duplicated().sum())
    print('Números armazenados como texto:', [c for c in obj if parece_numero(d[c]) > 0.9 and not ID_COL.search(c)])
    print('Marcadores de ausência em texto:', {c: int(d[c].astype(str).str.strip().str.lower().isin(MARCADORES - {''}).sum()) for c in obj if d[c].astype(str).str.strip().str.lower().isin(MARCADORES - {''}).any()})

In [ ]:
# FUNÇÕES DO PIPELINE
def achar_coluna(df, palavras, exclui=()):
    for c in df.columns:
        n = str(c).lower()
        if any(p in n for p in palavras) and not any(e in n for e in exclui):
            return c
    return None

def corrigir_tipos(df):
    """Preço, desconto e avaliações escritos como texto ('$12.99', '-30%', '1.2k') viram número."""
    df = df.copy()
    for c in df.select_dtypes(include=['object', 'string']).columns:
        n = str(c).lower()
        if ID_COL.search(n):
            continue
        if 'date' in n or 'time' in n:
            df[c] = pd.to_datetime(df[c], errors='coerce')
        elif parece_numero(df[c]) >= 0.9:
            limpo = limpar_numero(df[c])
            mult = limpo.str.contains(r'[kK]$', regex=True).fillna(False).map({True: 1000, False: 1})
            df[c] = pd.to_numeric(limpo.str.replace(',', '', regex=False).str.replace(r'[kK]$', '', regex=True), errors='coerce') * mult
    return df

def padronizar_textos(df):
    """Tira espaços extras, transforma marcadores (N/A, -, ?) em NaN e deixa categorias em minúsculas."""
    df = df.copy()
    for c in df.select_dtypes(include=['object', 'string']).columns:
        s = df[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
        s = s.mask(s.str.lower().isin(MARCADORES))
        if s.notna().sum() and s.nunique() / s.notna().sum() < 0.05:
            s = s.str.lower()
        df[c] = s.astype(object).where(s.notna(), np.nan)
    return df

def tratar_duplicatas(df):
    """Remove linhas repetidas e produtos repetidos (pelo identificador, se existir)."""
    df = df.drop_duplicates()
    for chave in ('product_id', 'goods_id', 'sku', 'url', 'id'):
        col = next((c for c in df.columns if str(c).lower() == chave), None)
        if col is not None:
            df = df.drop_duplicates(subset=[col])
            break
    return df.reset_index(drop=True)

def tratar_outliers(df, k=3.0):
    """Preço negativo vira NaN; desconto fica em módulo; extremos limitados pelo IQR (capping)."""
    df = df.copy()
    for c in df.select_dtypes(include='number').columns:
        n = str(c).lower()
        if ID_COL.search(n):
            continue
        if 'discount' in n:
            df[c] = df[c].abs()
        elif any(p in n for p in ('price', 'review', 'sold')):
            df.loc[df[c] < 0, c] = np.nan
        if df[c].nunique() > 20:
            q1, q3 = df[c].quantile([0.25, 0.75])
            if q3 > q1:
                df[c] = df[c].clip(q1 - k * (q3 - q1), q3 + k * (q3 - q1))
    return df

def tratar_ausentes(df):
    """Descarta colunas com mais de 60% vazias e linhas sem preço; mediana nas numéricas, 'desconhecido' nas de texto."""
    df = df.copy()
    descartadas = df.columns[df.isna().mean() > 0.60].tolist()
    df = df.drop(columns=descartadas)
    preco = achar_coluna(df, ['price'], exclui=('discount',))
    if preco is not None and pd.api.types.is_numeric_dtype(df[preco]):
        df = df.dropna(subset=[preco])
    for c in df.select_dtypes(include='number').columns:
        df[c] = df[c].fillna(df[c].median())
    for c in df.select_dtypes(include=['object', 'string']).columns:
        df[c] = df[c].fillna('desconhecido')
    df.attrs['descartadas'] = descartadas
    return df

def criar_features(df):
    """Cria desconto em %, flag de desconto e faixa de preço (quartis)."""
    df = df.copy()
    preco = achar_coluna(df, ['price'], exclui=('discount',))
    desc = achar_coluna(df, ['discount'])
    if desc is not None and pd.api.types.is_numeric_dtype(df[desc]):
        df['desconto_pct'] = df[desc] * 100 if df[desc].max() <= 1 else df[desc]
        df['tem_desconto'] = (df['desconto_pct'] > 0).astype(int)
    if preco is not None and pd.api.types.is_numeric_dtype(df[preco]):
        df['faixa_preco'] = pd.qcut(df[preco], q=4, labels=['barato', 'médio', 'caro', 'premium'], duplicates='drop')
    return df

def pipeline(df):
    # Ordem: texto -> tipos -> duplicatas -> outliers -> ausentes -> features
    return (df.pipe(padronizar_textos).pipe(corrigir_tipos).pipe(tratar_duplicatas)
              .pipe(tratar_outliers).pipe(tratar_ausentes).pipe(criar_features))

In [ ]:
# EXECUÇÃO DO PIPELINE
limpos = {nome: pipeline(d) for nome, d in brutos.items()}    # a mesma função serve para qualquer arquivo do dataset
for nome, d in limpos.items():
    print(nome, brutos[nome].shape, '->', d.shape, '| colunas descartadas:', d.attrs['descartadas'])

In [ ]:
# TABELA ANTES x DEPOIS
def indicadores(d):
    return {'linhas': len(d), '% ausentes': round(d.isna().sum().sum() / d.size * 100, 2),
            'duplicatas': int(d.duplicated().sum()),
            'colunas numéricas': d.select_dtypes(include='number').shape[1],
            'colunas texto': d.select_dtypes(include=['object', 'string']).shape[1]}

tabela = pd.concat({nome: pd.DataFrame({'antes': indicadores(brutos[nome]), 'depois': indicadores(limpos[nome])})
                    for nome in brutos}, axis=1)
display(tabela)

n0 = list(brutos)[0]
display(pd.DataFrame({'antes': brutos[n0].dtypes.astype(str), 'depois': limpos[n0].dtypes.astype(str)}))

In [ ]:
# VISUALIZAÇÕES
antes, depois = brutos[n0], limpos[n0]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# 1) % de ausentes por coluna
aus = pd.DataFrame({'antes': antes.isna().mean() * 100, 'depois': depois.isna().mean() * 100}).fillna(0)
aus[(aus['antes'] > 0) | (aus['depois'] > 0)].plot(kind='barh', ax=axes[0], title='% de ausentes por coluna')

# 2) distribuição do preço depois da limpeza
col_preco = achar_coluna(depois, ['price'], exclui=('discount',))
if col_preco:
    sns.histplot(depois[col_preco], bins=50, ax=axes[1]); axes[1].set_title(f'{col_preco} depois da limpeza')

# 3) resposta à pergunta de negócio: desconto médio por faixa de preço
if 'faixa_preco' in depois and 'desconto_pct' in depois:
    depois.groupby('faixa_preco', observed=True)['desconto_pct'].mean().plot(kind='bar', ax=axes[2], title='Desconto médio (%) por faixa de preço')
plt.tight_layout(); plt.show()

In [ ]:
# RELATÓRIO E RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO

# RELATÓRIO (decisões e justificativas; os números estão na tabela antes x depois acima)
# - Textos: tirei espaços extras e transformei marcadores (N/A, -, ?) em NaN; sem isso eles contam como categorias e o isna() não vê os ausentes.
# - Tipos: preço, desconto e avaliações escritos como texto ("$12.99", "-30%", "1.2k") viraram número, para poder calcular médias.
# - Duplicatas: removi linhas repetidas e produtos repetidos, para cada produto contar uma vez.
# - Outliers: preço negativo virou NaN e os extremos foram limitados pelo IQR (k=3, conservador), porque preços altos reais existem.
# - Ausentes: descartei colunas com mais de 60% vazias; removi linhas sem preço (não respondem à pergunta);
#   mediana nas numéricas e "desconhecido" nas de texto.
# - Features: desconto_pct, tem_desconto e faixa_preco, que respondem direto à pergunta de negócio.

# 1) Qual problema foi o mais difícil de perceber?
# Números escritos como texto e marcadores de ausência disfarçados ("N/A", "-"). O info() mostra "object" e o
# isna() mostra zero, então a base parece limpa. Só aparece ao tentar calcular uma média ou ao olhar os valores únicos.

# 2) A ordem das etapas importa?
# Sim. Se eu preencher os ausentes com a média antes de tratar os outliers, os valores extremos já puxaram a
# média e o valor preenchido carrega essa distorção para várias linhas. Por isso trato os outliers antes e uso a
# mediana. Também removo duplicatas antes de calcular medianas e limites, para repetições não pesarem nas contas.

# 3) Se a pergunta de negócio fosse outra, o que eu mudaria?
# Se a pergunta fosse achar preços anormais ou fraudulentos, eu não limitaria (capping) nem removeria os
# outliers de preço, porque eles seriam o objeto de estudo. Se fosse sobre volume de vendas, eu não removeria as
# linhas sem preço e trataria a coluna de vendas com mais cuidado.